# Lesson 21 — 资源所有权、Mutex 与何时不用 RTOS

输入端产生通知，网络端发送通知；先明确所有权，再考虑 Queue、Mutex、Semaphore 和 Event。这个 host lab 是确定性状态机；真实线程探针会确保 join，不会让学习环境永久死锁。

**学习顺序：** 先运行 guided，再修改，故意运行 broken，最后自己完成 exercise。答案 Notebook 独立保存。

In [ ]:
from pathlib import Path
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools/notebook_setup.py").is_file())
exec((project / "tools/notebook_setup.py").read_text())

## 1. Guided

先观察面板输出，再读 C++。

In [ ]:
%%academy_lab 21-resource-ownership guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& d):device_(d){} void tick(){if(device_.input.button&&!previous_)++pending_;previous_=device_.input.button;device_.output.error=device_.input.fault;if(device_.input.fault)device_.output.state="fault";else if(!device_.input.connected)device_.output.state="waiting-link";else if(pending_>0){--pending_;++device_.output.events;device_.output.outgoing="notification";device_.output.state="sent";}else device_.output.state="idle";device_.output.queue_depth=pending_;} private:Device& device_;bool previous_=false;int pending_=0;}; }

## 2. Modify

让网络恢复后每次 tick 只消费一条通知，观察离线期间 queue_depth 是否保留。

In [ ]:
%%academy_lab 21-resource-ownership modify
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& d):device_(d){} void tick(){if(device_.input.button&&!previous_)++pending_;previous_=device_.input.button;device_.output.error=device_.input.fault;if(device_.input.fault)device_.output.state="fault";else if(!device_.input.connected)device_.output.state="waiting-link";else if(pending_>0){--pending_;++device_.output.events;device_.output.outgoing="notification";device_.output.state="sent";}else device_.output.state="idle";device_.output.queue_depth=pending_;} private:Device& device_;bool previous_=false;int pending_=0;}; }

## 3. Broken

下面的检查应该失败；先观察故障证据，再查看分层提示。

In [ ]:
%%academy_lab 21-resource-ownership broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& d):device_(d){} void tick(){if(device_.input.button)++pending_;previous_=device_.input.button;if(device_.input.connected&&pending_>0){--pending_;++device_.output.events;}device_.output.queue_depth=pending_;} private:Device& device_;bool previous_=false;int pending_=0;}; }

## 4. Build

要求：一次按下只有一个通知；离线时保留；故障时不发送。然后回答：只有一个周期循环时，为什么可以不用 RTOS？

提示 1：电平和边沿不同。提示 2：输入端产生，发送端消费。提示 3：资源所有权比“多加一个 Mutex”更重要。

In [ ]:
%%academy_lab 21-resource-ownership exercise exercise
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& d):device_(d){} void tick(){ /* TODO: 你的实现 */ } private: Device& device_; }; }


## 6. 反思

写下：这个模型在 host 上证明了什么？哪些行为必须买板后重新验证？